# Arboretum Tree Segmentation - Surveyed Seeds + Dalponte 2016

Crowns are grown **from the in-person surveyed tree locations** instead of from automatically detected treetops. Each surveyed point is snapped to the highest smoothed-CHM cell within `snap_radius_m`, then used as a Dalponte seed. One surveyed tree = one crown.

## What each R step does

| Step | Function | What it does |
|---|---|---|
| Smoothing | `terra::focal(..., median)` | Replaces each cell by the median of its k x k neighbours; removes spikes/pits. |
| Seed placement | `snap_seeds()` (this notebook) | Moves each surveyed point (trunk base, possibly leaning or GPS-offset) to the highest CHM cell within `snap_radius_m`, so the seed sits on the crown top. |
| Crown Delineation | `lidR::dalponte2016(chm, seeds, th_tree, th_seed, th_cr, max_cr)` | Region growing from each seed. A neighbouring cell joins the crown if its height is > `th_tree`; > `th_seed` x the **seed** height; > `th_cr` x the **mean crown height so far**; and it lies within `max_cr` (pixels) of the seed. |

Canopy that has no surveyed tree gets no crown in the outputs. Unsurveyed treetops (found with `lmf`) are still used as **blocker seeds** during region growing, so surveyed crowns do not absorb their neighbours; the blocker crowns are discarded afterwards.

## 1. Packages

In [ ]:
# install.packages(c("terra", "sf", "lidR", "dplyr", "readxl"))
suppressPackageStartupMessages({ library(terra); library(sf); library(lidR); library(dplyr); library(readxl) })

## 2. Configuration

In [ ]:
cfg <- list(
  # --- CHM / Dalponte (starting values, retune for the arboretum) ---
  smooth_k   = 5,      # median smoothing kernel (cells). 5x5 @ 0.25 m = 1.25 m
  min_height = 1.5,    # seeds / crown cells below this height (m) are ignored
  max_height = 25,     # CHM cells above this (m) are set to NA
  th_tree    = 1.5,    # Dalponte: min height of any crown cell (m)
  th_seed    = 0.7,   # Dalponte: cell must be > th_seed x seed height
  th_cr      = 0.8,    # Dalponte: cell must be > th_cr x mean crown height
  # --- surveyed seed points ---
  blocker_ws        = 4.25, # lmf window (m) for finding UNSURVEYED treetops that compete with surveyed seeds
  blocker_min_dist_m = 3,   # lmf tops closer than this (m) to a surveyed seed are the same tree, not blockers
  peak_ws       = 2,   # window (m) for finding individual crown peaks that surveyed points can be matched to
  snap_radius_m = 3,   # surveyed point is matched to the NEAREST crown peak within this radius (m)
  survey_sheet  = "Biomass Calcs",
  col_id        = "Tree Num",
  col_x         = "Long",
  col_y         = "Lat",
  col_height    = "GT Height",
  col_dbh       = "GT DBH (m)",
  col_species   = "Sci. Species Name",
  col_footprint = "Footprint",   # matches reading_order_id in the footprint file; "4_6" = tree shared by footprints 4 and 6
  survey_crs    = 4326  # coordinates are lat/long in degrees (assumed WGS84)
)

## 3. Load CHM and smooth

In [ ]:
# Paths are relative to the notebooks/ folder
chm_path       <- "../../Rasters/arb_chm_0.25m_clipped.tif"
survey_path    <- "../data/WorkingTrees_Arboretum_Data.xlsx"
footprint_path <- "../../../CODE/Notebook_Test1_Arb_Output_Files/arb_bio_test_2_gedi_footprints_clipped.geojson"
out_dir        <- "../outputs/Arboretum Seeded Segmentation"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

chm <- rast(chm_path)
chm[chm > cfg$max_height] <- NA
res_m <- res(chm)[1]
cat("CHM:", basename(chm_path), "| resolution:", res_m, "m\n")

chm_smooth <- focal(chm, w = matrix(1, cfg$smooth_k, cfg$smooth_k), fun = median, na.rm = TRUE)

## 4. Load surveyed tree locations and snap them to crown tops

Each point is moved to the highest smoothed-CHM cell within `snap_radius_m`. Points with no canopy cell in range (or below `min_height`) are dropped and listed.

In [ ]:
survey <- read_excel(survey_path, sheet = cfg$survey_sheet, .name_repair = "minimal")
survey <- survey[!is.na(survey[[cfg$col_x]]) & !is.na(survey[[cfg$col_y]]) & !is.na(survey[[cfg$col_id]]), ]
# ignore trees with no height listed in the spreadsheet
no_height <- is.na(suppressWarnings(as.numeric(survey[[cfg$col_height]])))
cat("Ignored", sum(no_height), "tree(s) with no", cfg$col_height, "value\n")
survey <- survey[!no_height, ]

survey_pts <- st_as_sf(
  data.frame(tree_num    = as.character(survey[[cfg$col_id]]),
             species     = as.character(survey[[cfg$col_species]]),
             footprint   = as.character(survey[[cfg$col_footprint]]),
             gt_height_m = suppressWarnings(as.numeric(survey[[cfg$col_height]])),
             gt_dbh_m    = suppressWarnings(as.numeric(survey[[cfg$col_dbh]])),
             lon         = as.numeric(survey[[cfg$col_x]]),
             lat         = as.numeric(survey[[cfg$col_y]])),
  coords = c("lon", "lat"), crs = cfg$survey_crs, remove = FALSE) |>
  st_transform(crs(chm))

if (anyDuplicated(survey_pts$tree_num)) warning("Duplicate Tree Num values: ", paste(unique(survey_pts$tree_num[duplicated(survey_pts$tree_num)]), collapse = ", "))

# Highest smoothed-CHM cell within radius r (m) of each point
snap_seeds <- function(pts, chm, r) {
  xy <- st_coordinates(pts)
  out <- lapply(seq_len(nrow(xy)), function(i) {
    x <- xy[i, 1]; y <- xy[i, 2]
    w <- tryCatch(crop(chm, ext(x - r, x + r, y - r, y + r)), error = function(e) NULL)
    if (is.null(w)) return(c(sx = NA, sy = NA, sh = NA, snap_dist_m = NA))
    cc <- xyFromCell(w, seq_len(ncell(w))); v <- values(w)[, 1]
    d  <- sqrt((cc[, 1] - x)^2 + (cc[, 2] - y)^2)
    ok <- which(d <= r & !is.na(v))
    if (!length(ok)) return(c(sx = NA, sy = NA, sh = NA, snap_dist_m = NA))
    j <- ok[which.max(v[ok])]
    c(sx = unname(cc[j, 1]), sy = unname(cc[j, 2]), sh = unname(v[j]), snap_dist_m = unname(d[j]))
  })
  cbind(st_drop_geometry(pts), as.data.frame(do.call(rbind, out)))
}

snapped <- snap_seeds(survey_pts, chm_smooth, cfg$snap_radius_m)

# Match each surveyed point to the NEAREST real crown peak within the radius, by distance only
# (height is not used, so the height comparison stays honest). If no peak is in range,
# fall back to the tallest cell in range, as before.
peaks <- locate_trees(chm_smooth, lmf(ws = cfg$peak_ws, hmin = cfg$min_height))
pk_h  <- st_coordinates(peaks)[, "Z"]
peaks <- st_zm(peaks)
d_pk  <- matrix(as.numeric(st_distance(survey_pts, peaks)), nrow = nrow(survey_pts))
j     <- apply(d_pk, 1, which.min)
dmin  <- d_pk[cbind(seq_len(nrow(d_pk)), j)]
use_pk <- dmin <= cfg$snap_radius_m
pk_xy  <- st_coordinates(peaks)
snapped$snap_method <- ifelse(use_pk, "nearest peak", "tallest cell")
snapped$sx[use_pk] <- pk_xy[j[use_pk], 1]
snapped$sy[use_pk] <- pk_xy[j[use_pk], 2]
snapped$sh[use_pk] <- pk_h[j[use_pk]]
snapped$snap_dist_m[use_pk] <- dmin[use_pk]
cat(sprintf("Matched to nearest peak: %d | fell back to tallest cell: %d\n", sum(use_pk), sum(!use_pk)))
no_canopy <- is.na(snapped$sx) | snapped$sh < cfg$min_height
cell_id   <- cellFromXY(chm_smooth, cbind(snapped$sx, snapped$sy))
collide   <- !no_canopy & duplicated(cell_id)       # two surveyed trees snapped to the same cell

cat(sprintf("Surveyed points: %d | dropped (no canopy >= %.1f m within %.1f m): %d | dropped (same cell as another seed): %d\n",
            nrow(snapped), cfg$min_height, cfg$snap_radius_m, sum(no_canopy), sum(collide)))
if (any(no_canopy)) cat("No canopy:", paste(snapped$tree_num[no_canopy], collapse = ", "), "\n")
if (any(collide))   cat("Same cell as another seed:", paste(snapped$tree_num[collide], collapse = ", "), "\n")
cat(sprintf("Snap distance: median %.2f m, max %.2f m\n",
            median(snapped$snap_dist_m[!no_canopy], na.rm = TRUE), max(snapped$snap_dist_m[!no_canopy], na.rm = TRUE)))

keep  <- !no_canopy & !collide
ttops <- st_as_sf(snapped[keep, ], coords = c("sx", "sy"), crs = st_crs(survey_pts), remove = FALSE)
ttops$treeID <- seq_len(nrow(ttops))
ttops$height <- ttops$sh
cat("Seeds used:", nrow(ttops), "\n")

# Blocker seeds: unsurveyed treetops found with lmf. They compete with surveyed seeds during region growing
# so surveyed crowns stop absorbing neighbouring canopy; their crowns are discarded afterwards.
lmf_tops <- locate_trees(chm_smooth, lmf(ws = cfg$blocker_ws, hmin = cfg$min_height)) |> st_zm()
near     <- apply(st_distance(lmf_tops, ttops), 1, min) < cfg$blocker_min_dist_m
blockers <- lmf_tops[!near, ]
blockers$treeID <- nrow(ttops) + seq_len(nrow(blockers))
all_seeds <- st_sf(treeID = c(ttops$treeID, blockers$treeID),
                   geometry = c(st_geometry(ttops), st_geometry(blockers)))
cat("Blocker seeds (unsurveyed treetops):", nrow(blockers), "\n")


### Snap diagnostic

Seeds that moved almost the full `snap_radius_m` (>= 90%) probably have a true crown top outside the search radius, or snapped onto a taller neighbour. The map draws a line from each surveyed point (black x) to its seed (red +); red lines are edge snaps.

In [ ]:
snap_df <- ttops |> st_drop_geometry() |>
  mutate(at_edge = snap_dist_m >= 0.9 * cfg$snap_radius_m,
         seed_minus_gt_m = height - gt_height_m)
cat(sprintf("Seeds snapped to >= 90%% of the radius: %d of %d
", sum(snap_df$at_edge), nrow(snap_df)))
cat(sprintf("Seed height minus ground height: median %.2f m | trees > +3 m: %d
",
            median(snap_df$seed_minus_gt_m, na.rm = TRUE), sum(snap_df$seed_minus_gt_m > 3, na.rm = TRUE)))
print(snap_df |> filter(at_edge | seed_minus_gt_m > 3) |>
        select(treeID, tree_num, snap_dist_m, seed_h = height, gt_height_m, seed_minus_gt_m) |>
        arrange(desc(snap_dist_m)), digits = 3)

hist(snap_df$snap_dist_m, breaks = 20, xlim = c(0, cfg$snap_radius_m), col = "grey80",
     main = "Snap distance (surveyed point to seed)", xlab = "m")

sp <- survey_pts[match(ttops$tree_num, survey_pts$tree_num), ]
e  <- ext(vect(st_geometry(ttops))) + 5
plot(crop(chm_smooth, e), range = c(0, cfg$max_height), col = height.colors(50), main = "Surveyed point (x) -> seed (+)")
segments(st_coordinates(sp)[, 1], st_coordinates(sp)[, 2], ttops$sx, ttops$sy,
         col = ifelse(snap_df$at_edge, "red", "black"), lwd = 1.5)
points(st_coordinates(sp), pch = 4, col = "black", cex = 0.8)
points(ttops$sx, ttops$sy, pch = 3, col = "red", cex = 0.8)

### Height-guided snap (diagnostic only)

For each tree, finds the cell within `snap_radius_m` whose smoothed CHM height is **closest to the surveyed height**, and compares it with the seed actually used. This does **not** change the seeds or crowns. If even the best-matching cell is far from the surveyed height, the problem is the surveyed point or height (location error, bad field value), not the snapping rule. If a close match exists but is a different cell, the seed chosen by distance is on a neighbour.

In [ ]:
closest_cell <- function(x, y, gt_h, chm, r) {
  w  <- crop(chm, ext(x - r, x + r, y - r, y + r))
  cc <- xyFromCell(w, seq_len(ncell(w))); v <- values(w)[, 1]
  d  <- sqrt((cc[, 1] - x)^2 + (cc[, 2] - y)^2)
  ok <- which(d <= r & !is.na(v))
  if (!length(ok) || is.na(gt_h)) return(c(NA, NA))
  j <- ok[which.min(abs(v[ok] - gt_h))]
  c(unname(v[j]), unname(d[j]))
}

sp  <- survey_pts[match(ttops$tree_num, survey_pts$tree_num), ]
xy  <- st_coordinates(sp)
res <- t(sapply(seq_len(nrow(ttops)), function(i) closest_cell(xy[i, 1], xy[i, 2], ttops$gt_height_m[i], chm_smooth, cfg$snap_radius_m)))

guided <- ttops |> st_drop_geometry() |>
  transmute(tree_num, species = substr(species, 1, 28), footprint, gt_h = round(gt_height_m, 2),
            seed_h = round(height, 2), seed_dist_m = round(snap_dist_m, 2),
            closest_h = round(res[, 1], 2), closest_dist_m = round(res[, 2], 2),
            seed_err = round(height - gt_height_m, 2), closest_err = round(res[, 1] - gt_height_m, 2))

options(width = 250)
cat(sprintf("Trees with a cell within %.1f m of the surveyed height (|error| <= 1 m): %d of %d%s",
            cfg$snap_radius_m, sum(abs(guided$closest_err) <= 1, na.rm = TRUE), sum(!is.na(guided$closest_err)), "  (seed used: "),
    sum(abs(guided$seed_err) <= 1, na.rm = TRUE), ")\n", sep = "")
cat("Trees with seed error > 3 m, largest first:\n")
print(guided |> filter(abs(seed_err) > 3) |> arrange(desc(abs(seed_err))), row.names = FALSE)

## 5. Dalponte crown delineation

In [ ]:
max_cr_px <- max(nrow(chm_smooth), ncol(chm_smooth))   # no crown size limit: window as large as the raster (lidR requires a value, in pixels)
crowns <- dalponte2016(chm_smooth, all_seeds, th_tree = cfg$th_tree, th_seed = cfg$th_seed,
                       th_cr = cfg$th_cr, max_cr = max_cr_px)()
cat("max_cr =", max_cr_px, "pixels (effectively unlimited)\n")

names(crowns) <- "treeID"
polys <- st_as_sf(as.polygons(crowns, dissolve = TRUE))
polys <- polys[polys$treeID <= nrow(ttops), ]      # drop blocker crowns, keep surveyed trees only
polys$area_m2 <- as.numeric(st_area(polys))
polys$diam_m  <- 2 * sqrt(polys$area_m2 / pi)     # equivalent circular diameter
cat("Crowns:", nrow(polys), "of", nrow(ttops), "seeds | median crown diameter:", round(median(polys$diam_m), 2), "m\n")

## 6. Visualization

In [ ]:
footprints <- st_read(footprint_path, quiet = TRUE) |> st_transform(st_crs(chm))

plot(chm, range = c(0, cfg$max_height), col = height.colors(50))
plot(st_geometry(polys), border = "white", col = NA, lwd = 0.5, add = TRUE)
plot(st_geometry(survey_pts), pch = 4, col = "black", cex = 0.6, add = TRUE)        # surveyed point (x)
plot(st_geometry(ttops), pch = 3, col = "red", cex = 0.6, add = TRUE)               # snapped seed (+)
plot(st_geometry(footprints), border = "red", col = NA, lwd = 3, add = TRUE)

### Per-footprint views

In [ ]:
# Distance from each surveyed coordinate to the CHM max (seed) matched to it
# dist_m = distance (m); chm_h = CHM max (seed) height; gt_h = ground height; diff_h = chm_h - gt_h
snap_list <- ttops |> st_drop_geometry() |>
  transmute(tree_num, species = substr(species, 1, 28), footprint,
            dist_m = round(snap_dist_m, 2),
            chm_h  = round(height, 2),
            gt_h   = round(gt_height_m, 2),
            diff_h = round(height - gt_height_m, 2))   # same row order as ttops

# One zoomed plot per GEDI footprint. Trees are assigned by the survey's Footprint column
# (= reading_order_id in the footprint file), so trees just outside the polygon still show up.
pad <- 5
options(width = 250)   # keep the per-footprint tables on one line
options(repr.plot.width = 12, repr.plot.height = 10)
for (i in order(footprints$reading_order_id)) {
  fp  <- footprints[i, ]
  id  <- as.character(fp$reading_order_id)
  in_fp <- !is.na(ttops$footprint) & sapply(strsplit(ttops$footprint, "_"), function(z) id %in% z)
  e   <- ext(vect(st_geometry(fp)))
  if (any(in_fp)) e <- e + ext(vect(st_geometry(ttops[in_fp, ])))   # union of footprint + its trees
  e   <- e + pad
  plot(crop(chm_smooth, e), range = c(0, cfg$max_height), col = height.colors(50),
       main = sprintf("Footprint %s: %d seeded trees", id, sum(in_fp)))
  plot(st_geometry(polys[polys$treeID %in% ttops$treeID[in_fp], ]), border = "white", col = NA, lwd = 2, add = TRUE)
  plot(st_geometry(fp), border = "yellow", col = NA, lwd = 2, add = TRUE)
  sp <- survey_pts[match(ttops$tree_num[in_fp], survey_pts$tree_num), ]
  segments(st_coordinates(sp)[, 1], st_coordinates(sp)[, 2], ttops$sx[in_fp], ttops$sy[in_fp], col = "black")
  points(st_coordinates(sp), pch = 4, col = "black", lwd = 2, cex = 1.3)
  points(ttops$sx[in_fp], ttops$sy[in_fp], pch = 3, col = "red", lwd = 2, cex = 1.3)
  text(ttops$sx[in_fp], ttops$sy[in_fp], ttops$tree_num[in_fp], pos = 3, col = "white", cex = 0.9, font = 2)
  cat("\nFootprint ", id, ": distance from each surveyed tree to its CHM max (largest first)\n", sep = "")
  print(snap_list[in_fp, ] |> arrange(desc(dist_m)), row.names = FALSE)
}

## 7. Compare to ground-measured heights

Matched by **Tree Num** (no distance matching). Two segmented heights are compared to the surveyed `GT Height`: the smoothed-CHM seed height and the maximum raw-CHM value inside the crown (smoothing lowers peaks).

In [ ]:
height_metrics <- function(est, obs) {
  e <- est - obs
  c(n = length(e), bias = mean(e), MAE = mean(abs(e)), RMSE = sqrt(mean(e^2)),
    R2 = suppressWarnings(cor(est, obs)^2))
}

raw_max <- terra::extract(chm, vect(polys), fun = max, na.rm = TRUE)[, 2]

comparison <- ttops |> st_drop_geometry() |>
  left_join(data.frame(treeID = polys$treeID, crown_diam_m = polys$diam_m, crown_area_m2 = polys$area_m2, raw_max_m = raw_max),
            by = "treeID") |>
  transmute(tree_num, species, gt_height_m, seg_height_m = height, raw_max_m,
            err_seg = seg_height_m - gt_height_m, err_raw = raw_max_m - gt_height_m,
            snap_dist_m, crown_diam_m, crown_area_m2, treeID)

ok <- !is.na(comparison$gt_height_m) & !is.na(comparison$raw_max_m)
metrics <- rbind(`smoothed treetop height` = height_metrics(comparison$seg_height_m[ok], comparison$gt_height_m[ok]),
                 `raw CHM max in crown`    = height_metrics(comparison$raw_max_m[ok],    comparison$gt_height_m[ok]))
cat(sprintf("%d of %d seeded trees have a ground height\n", sum(ok), nrow(comparison)))
print(round(metrics, 3))

print(comparison, digits = 3)
write.csv(comparison, file.path(out_dir, "ground_height_comparison.csv"), row.names = FALSE)

## 8. Export

In [ ]:
tree_data <- ttops |> st_drop_geometry() |>
  transmute(treeID, tree_num, species, footprint, x = sx, y = sy, height_m = height, snap_dist_m, gt_height_m, gt_dbh_m) |>
  left_join(st_drop_geometry(polys) |> select(treeID, crown_area_m2 = area_m2, crown_diam_m = diam_m), by = "treeID")
write.csv(tree_data, file.path(out_dir, "tree_heights.csv"), row.names = FALSE)
st_write(polys,  file.path(out_dir, "crowns.gpkg"),   delete_dsn = TRUE, quiet = TRUE)
st_write(ttops,  file.path(out_dir, "treetops.gpkg"), delete_dsn = TRUE, quiet = TRUE)
cat("Saved to", normalizePath(out_dir), "\n")